# 🌦️ Rainfall Prediction & Meteorological Classification Pipeline

**Author:** Dickson Madueke  
**Objective:** Predict daily precipitation occurrence and Probability of Precipitation (PoP) from terrestrial atmospheric observations.  
**Target:** Binary Classification (`0` = No Rain, `1` = Rain)  

---

## 1. Project Overview & Meteorological Context
Precipitation forecasting is fundamental to agricultural planning, flood risk management, urban drainage logistics, and aviation safety.
In this study, we build a robust, reproducible machine learning pipeline that:
1. Ingests and sanitizes 366 daily meteorological observations.
2. Resolves data leakage in missing value imputation.
3. Scales features to prevent atmospheric pressure from distorting regression weights.
4. Compares multiple machine learning algorithms using Stratified 5-Fold Cross-Validation.
5. Exports a production-grade scikit-learn pipeline for deployment on Streamlit Cloud.


In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import joblib

print("Libraries successfully imported.")


## 2. Data Ingestion & Schema Cleaning
We load the raw dataset `dataset1.csv` and sanitize column headers (stripping whitespace and normalizing names).


In [2]:
data_path = os.path.join("..", "dataset1.csv") if not os.path.exists("dataset1.csv") else "dataset1.csv"
raw_df = pd.read_csv(data_path)
print(f"Raw dataset shape: {raw_df.shape}")
print("Original columns:", raw_df.columns.tolist())

# Standardize columns
df = raw_df.copy()
df.columns = [c.strip().lower() for c in df.columns]
df = df.rename(columns={"temparature": "temperature", "winddirection": "winddirection"})
print("Cleaned columns:", df.columns.tolist())
df.head()


## 3. Exploratory Data Analysis & Target Balance
We inspect class distribution and missingness patterns.


In [3]:
print("Missing value count per column:")
print(df.isnull().sum())

# Target encoding
df["rainfall_binary"] = df["rainfall"].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})
target_counts = df["rainfall_binary"].value_counts()
print("\nTarget Distribution:")
print(target_counts)


## 4. Feature Selection & Pipeline Construction
We define the 9 core meteorological predictors, dropping index/date column `day` and redundant `temperature`.


In [4]:
feature_cols = ["pressure", "maxtemp", "mintemp", "dewpoint", "humidity", "cloud", "sunshine", "winddirection", "windspeed"]
X = df[feature_cols].copy()
y = df["rainfall_binary"].copy()

print(f"Feature matrix X: {X.shape}")
print(f"Target vector y: {y.shape}")


## 5. Stratified 5-Fold Cross Validation Benchmark
We evaluate 4 distinct architectures: Logistic Regression (with StandardScaler), Random Forest, Gradient Boosting, and Decision Tree.


In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    "Logistic Regression (Standardized)": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, max_depth=3, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42)
}

benchmark_summary = []
for name, clf in models.items():
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if "Standardized" in name or "Logistic" in name:
        steps.append(("scaler", StandardScaler()))
    steps.append(("classifier", clf))
    pipe = Pipeline(steps)

    accs, precs, recs, f1s, aucs = [], [], [], [], []
    for tr_idx, te_idx in skf.split(X, y):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
        pipe.fit(X_tr, y_tr)
        preds = pipe.predict(X_te)
        probs = pipe.predict_proba(X_te)[:, 1]
        accs.append(accuracy_score(y_te, preds))
        precs.append(precision_score(y_te, preds, zero_division=0))
        recs.append(recall_score(y_te, preds, zero_division=0))
        f1s.append(f1_score(y_te, preds, zero_division=0))
        aucs.append(roc_auc_score(y_te, probs))

    benchmark_summary.append({
        "Model": name,
        "Accuracy": np.mean(accs),
        "Precision": np.mean(precs),
        "Recall": np.mean(recs),
        "F1-Score": np.mean(f1s),
        "ROC-AUC": np.mean(aucs)
    })

results_df = pd.DataFrame(benchmark_summary).set_index("Model")
results_df.round(3)


## 6. Champion Model Holdout Evaluation
We evaluate the standardized Logistic Regression and Random Forest on an independent 20% holdout split.


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
champion_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])
champion_pipe.fit(X_train, y_train)
y_pred = champion_pipe.predict(X_test)
y_prob = champion_pipe.predict_proba(X_test)[:, 1]

print("=== Classification Report (Holdout Test) ===")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


## 7. Model Serialization & Production Export
We fit the production pipeline on the full dataset and serialize it with `joblib`.


In [7]:
champion_pipe.fit(X, y)
os.makedirs("../models", exist_ok=True)
model_out_path = os.path.join("..", "models", "rainfall_pipeline.joblib")
joblib.dump(champion_pipe, model_out_path)
print(f"Production pipeline saved to {model_out_path}")
